# 📊 Measures of Association in Epidemiology
### RR · RD · OR · NNT · IRR · IRD
##### Desy Nuryunarsih — Tutorial Series: Epidemiology with Python

---

## 🎯 What will you learn?

In this tutorial, you will learn **six fundamental measures** used in epidemiology to describe the relationship between an **exposure** and an **outcome**:

| Measure | Full Name | One-line meaning |
|---|---|---|
| **RR** | Risk Ratio | How many times more likely? |
| **RD** | Risk Difference | What is the absolute difference in risk? |
| **OR** | Odds Ratio | What is the ratio of odds? |
| **NNT** | Number Needed to Treat | How many people are affected per intervention? |
| **IRR** | Incidence Rate Ratio | RR when we account for follow-up time |
| **IRD** | Incidence Rate Difference | RD when we account for follow-up time |

> ⚠️ **Important note:** We call these *measures of association*, not measures of effect — because observational data alone cannot prove causation.

---

## 🚬 Our Story: Smoking and Lung Cancer

We will use a **simulated dataset** of 600 individuals followed for up to 52 weeks.

- **Exposure (A):** Smoking status → `A = 1` (smoker), `A = 0` (non-smoker)
- **Outcome (Y):** Lung cancer diagnosis → `Y = 1` (diagnosed), `Y = 0` (not diagnosed)
- **Time (T):** Weeks of follow-up per individual

We will ask: *Is smoking associated with a higher risk of lung cancer?*

---
## ⚙️ Step 1: Install and Import Libraries

We will use **zEpid** — a Python package specifically designed for epidemiological calculations.

Run the cell below to install it (only needed once in Google Colab).

In [ ]:
# Install zEpid (run this in Google Colab or if not yet installed)
!pip install zepid

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import zepid
from zepid import (RiskRatio, RiskDifference, OddsRatio, NNT,
                   IncidenceRateRatio, IncidenceRateDifference)

print('✅ zEpid version:', zepid.__version__)
print('✅ All libraries imported successfully!')

---
## 🗃️ Step 2: Create the Simulated Dataset

We simulate a cohort of **600 people** — 300 smokers and 300 non-smokers.

Based on published epidemiological evidence:
- Smokers have approximately **30% risk** of lung cancer over the study period
- Non-smokers have approximately **10% risk** over the same period

We also add **20 missing values** in the outcome — this is realistic! Real data always has some missing observations.

In [ ]:
# Set a random seed so results are reproducible every time you run this
np.random.seed(42)

# --- Smokers (exposure = 1) ---
smokers = pd.DataFrame({
    'smoking':         [1] * 300,
    'lung_cancer':     np.random.binomial(1, 0.30, 300),   # 30% risk
    'follow_up_weeks': np.random.randint(20, 52, 300)
})

# --- Non-smokers (exposure = 0) ---
nonsmokers = pd.DataFrame({
    'smoking':         [0] * 300,
    'lung_cancer':     np.random.binomial(1, 0.10, 300),   # 10% risk
    'follow_up_weeks': np.random.randint(20, 52, 300)
})

# Combine into one dataset
df = pd.concat([smokers, nonsmokers], ignore_index=True)

# Introduce 20 missing values in lung_cancer (realistic!)
missing_idx = np.random.choice(df.index, size=20, replace=False)
df.loc[missing_idx, 'lung_cancer'] = np.nan

print('📋 Dataset created!')
print(f'   Total participants: {len(df)}')
print(f'   Smokers: {df.smoking.sum()}')
print(f'   Non-smokers: {(df.smoking == 0).sum()}')
print()
df.info()

---
## 🔍 Step 3: Explore the Data

In [ ]:
# Preview the first 10 rows
print('First 10 rows of our dataset:')
df.head(10)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print()
print('💡 There are 20 missing observations in lung_cancer.')
print('   zEpid will automatically exclude them and report the count.')

In [ ]:
# The 2x2 table — the foundation of all our measures!
print('2×2 Contingency Table:')
print()
ct = pd.crosstab(
    df['smoking'].map({1: 'Smoker', 0: 'Non-smoker'}),
    df['lung_cancer'].map({1.0: 'Lung Cancer = YES', 0.0: 'Lung Cancer = NO'}),
    margins=True
)
print(ct)
print()
print('This table is the foundation of ALL the measures we will calculate!')

---
## 1️⃣ Risk Ratio (RR)

### 💡 Plain English first:
> *"How many times more likely are smokers to get lung cancer, compared to non-smokers?"

### 📐 Formula:

$$RR = \frac{Pr(Y=1 \mid A=1)}{Pr(Y=1 \mid A=0)} = \frac{\text{Risk in smokers}}{\text{Risk in non-smokers}}$$

### 🧭 How to interpret:
- **RR = 1.0** → No association (same risk in both groups)
- **RR > 1.0** → Exposure *increases* risk (smokers have higher risk)
- **RR < 1.0** → Exposure *decreases* risk (smokers have lower risk — unlikely here!)

In [ ]:
rr = RiskRatio()
rr.fit(df, exposure='smoking', outcome='lung_cancer')
rr.summary()

### ✏️ How to write this up:

> *"Smokers were approximately **2.7 times more likely** to develop lung cancer compared to non-smokers (RR = 2.72, 95% CI: 1.88, 3.94)."

Since the 95% confidence interval does **not** cross 1.0, this association is statistically significant.

---
## 2️⃣ Risk Difference (RD)

### 💡 Plain English first:
> *"What is the extra absolute risk of lung cancer that smokers carry, compared to non-smokers?"

### 📐 Formula:

$$RD = Pr(Y=1 \mid A=1) - Pr(Y=1 \mid A=0) = \text{Risk in smokers} - \text{Risk in non-smokers}$$

### 🧭 How to interpret:
- **RD = 0** → No difference in absolute risk
- **RD > 0** → Smokers have higher absolute risk
- **RD < 0** → Smokers have lower absolute risk

> 💬 **RR vs RD:** RR is a *relative* measure ("times more likely"), RD is an *absolute* measure ("percentage points more"). Both tell different stories!

In [ ]:
rd = RiskDifference()
rd.fit(df, exposure='smoking', outcome='lung_cancer')
rd.summary()

### ✏️ How to write this up:

> *"The risk of lung cancer was **19 percentage points higher** among smokers compared to non-smokers (RD = 0.19, 95% CI: 0.13, 0.25)."

💡 **Notice the LowerBound and UpperBound columns** — these are called *Fréchet probability bounds*. They always span a width of 1.0, and they represent the range of possible risk differences without needing any assumptions about exchangeability between groups.

---
## 3️⃣ Odds Ratio (OR)

### 💡 Plain English first:
> *"What is the ratio of the odds of getting lung cancer in smokers vs non-smokers?"

First — what is an **odd**? 
> Odds = (probability of event) ÷ (probability of no event)
> 
> If risk = 30%, then odds = 0.30 / 0.70 = 0.43

### 📐 Formula:

$$OR = \frac{Pr(Y=1 \mid A=1) / Pr(Y=0 \mid A=1)}{Pr(Y=1 \mid A=0) / Pr(Y=0 \mid A=0)}$$

### 🧭 When is OR used?
- **Case-control studies** — when you cannot directly calculate risk
- **Logistic regression** — the default output is an OR
- When the outcome is **rare** (<10%), OR ≈ RR

In [ ]:
oddr = OddsRatio()
oddr.fit(df, exposure='smoking', outcome='lung_cancer')
oddr.summary()

### ✏️ How to write this up:

> *"The odds of lung cancer were **3.5 times higher** among smokers compared to non-smokers (OR = 3.46, 95% CI: 2.21, 5.39)."

💡 **Notice:** OR (3.46) > RR (2.72). This is expected! When the outcome is **common** (>10%), OR overestimates the RR. When the outcome is rare (<10%), they are nearly equal.


---
## 4️⃣ Number Needed to Treat/Harm (NNT / NNH)

### 💡 Plain English first:
> *"If we could stop people from smoking, how many people would need to quit smoking to prevent ONE case of lung cancer?"

In this context, since smoking *causes* harm, we call this the **Number Needed to Harm (NNH)** — the number of people who smoke to produce one *extra* case of lung cancer.

### 📐 Formula:

$$NNT = \frac{1}{|RD|}$$

### ⚠️ Important:
NNT **implies causation**. You should only use it when you believe the association is truly causal.

In [ ]:
nnt = NNT()
nnt.fit(df, exposure='smoking', outcome='lung_cancer')
nnt.summary()

### ✏️ How to write this up:

> *"For every **~5 smokers**, one additional lung cancer case is attributable to smoking (NNH = 5.27, 95% CI: NNT 7.95 to NNH 3.94)."

💡 **The confidence interval spans from NNT to NNH.** This happens when the confidence interval of the risk difference crosses zero — meaning we cannot rule out that smoking could theoretically be protective (at the extreme end of the CI).

---
## 5️⃣ Incidence Rate Ratio (IRR)

### 💡 Plain English first:
> *"How many times higher is the rate of lung cancer diagnosis per week of follow-up among smokers vs non-smokers?"

### ❓ Why do we need a rate when we already have a risk?

- **Risk** = number of events / number of people → ignores *how long* people were followed
- **Rate** = number of events / total person-time → accounts for *how long* each person contributed

If some people were only followed for 20 weeks while others for 52 weeks, using a rate is fairer!

### 📐 Formula:

$$IRR = \frac{a / T_1}{b / T_0}$$

Where:
- $a$ = lung cancer cases in smokers, $T_1$ = total person-weeks in smokers
- $b$ = lung cancer cases in non-smokers, $T_0$ = total person-weeks in non-smokers

In [ ]:
irr = IncidenceRateRatio()
irr.fit(df, exposure='smoking', outcome='lung_cancer', time='follow_up_weeks')
irr.summary()

### ✏️ How to write this up:

> *"Smokers had an incidence rate of lung cancer **2.69 times higher** per person-week of follow-up compared to non-smokers (IRR = 2.69, 95% CI: 1.79, 4.03)."

💡 Notice the table now shows **Person-time** instead of just counts — this is the key difference from RR.

---
## 6️⃣ Incidence Rate Difference (IRD)

### 💡 Plain English first:
> *"What is the absolute difference in lung cancer rates per person-week between smokers and non-smokers?"

### 📐 Formula:

$$IRD = \frac{a}{T_1} - \frac{b}{T_0}$$

This is simply the **difference** between the two incidence rates — the person-time equivalent of the Risk Difference.

### 🌍 Public health use:
IRD is especially useful in vaccine studies — it directly shows the **vaccine-attributable reduction** in disease incidence.

In [ ]:
ird = IncidenceRateDifference()
ird.fit(df, exposure='smoking', outcome='lung_cancer', time='follow_up_weeks')
ird.summary()

### ✏️ How to write this up:

> *"Smokers had **0.005 additional lung cancer cases per person-week** compared to non-smokers (IRD = 0.005, 95% CI: 0.003, 0.007)."

This means: for every 1,000 person-weeks of follow-up, smoking produces approximately **5 extra lung cancer cases**.

---
## 📋 Summary: All Six Measures Side by Side

Here is a full comparison of all measures calculated in this tutorial:

In [ ]:
summary = pd.DataFrame({
    'Measure': ['Risk Ratio (RR)', 'Risk Difference (RD)', 'Odds Ratio (OR)',
                'Number Needed to Harm (NNH)', 'Incidence Rate Ratio (IRR)', 'Incidence Rate Difference (IRD)'],
    'Estimate': ['2.72', '0.19', '3.46', '5.27', '2.69', '0.005'],
    '95% CI': ['(1.88, 3.94)', '(0.13, 0.25)', '(2.21, 5.39)',
               '(NNT 7.95 to NNH 3.94)', '(1.79, 4.03)', '(0.003, 0.007)'],
    'Plain English': [
        'Smokers are 2.7× more likely to get lung cancer',
        'Smokers have 19 percentage points higher absolute risk',
        'Odds of lung cancer are 3.5× higher in smokers',
        '~5 smokers produce 1 extra lung cancer case',
        'Lung cancer rate per week is 2.7× higher in smokers',
        '5 extra cases per 1,000 person-weeks in smokers'
    ]
})

print('=' * 90)
print('SUMMARY TABLE: Measures of Association — Smoking and Lung Cancer')
print('=' * 90)
print(summary.to_string(index=False))
print()
print('✅ All estimates indicate a positive association between smoking and lung cancer.')

---
## 🎓 Key Takeaways

| Concept | Remember this |
|---|---|
| **RR vs OR** | Use RR for cohort studies; use OR for case-control studies or logistic regression |
| **RR vs RD** | RR tells you *relative* risk; RD tells you *absolute* risk — both matter! |
| **NNT / NNH** | The most clinically meaningful measure — but only valid when association = causation |
| **Rates vs Risks** | Use IRR/IRD when individuals have different follow-up times |
| **Confidence interval** | If CI crosses the null (1 for ratios, 0 for differences) → not significant |

---

## 📚 References

- Zivich PN et al. *zEpid: An epidemiology toolbox in Python.* JOSS, 2022.
- Rothman KJ, Greenland S, Lash TL. *Modern Epidemiology*, 3rd edition.
- Altman DG. *Confidence intervals for the number needed to treat.* BMJ, 1998.

---
*Tutorial by Desy Nuryunarsih | Research Fellow, University of St Andrews*